In [0]:
from pyspark.sql.functions import col, when, year, month, dayofweek, hour

SILVER_SALES_TABLE = "workspace.retail.silver_sales"
GOLD_FACT_SALES = "workspace.retail.gold_fact_sales"

def build_fact_sales(df_sales):
    
    df_fact = df_sales.withColumn(
        "is_return", 
        when(col("invoice").startswith("C"), True).otherwise(False)
    ).withColumn(
        "sales_year", year(col("invoice_date_only"))
    ).withColumn(
        "sales_month", month(col("invoice_date_only"))
    ).withColumn(
        "sales_day_of_week", dayofweek(col("invoice_date_only"))
    ).withColumn(
        "sales_hour", hour(col("invoice_date")) 
    )
    
   
    final_columns = [
        "invoice",              
        "customer_id",           
        "stock_code",           
        "invoice_date",         
        "invoice_date_only",    
        "sales_year", 
        "sales_month", 
        "sales_day_of_week", 
        "sales_hour",
        "quantity",             
        "price",                
        "total_amount",         
        "is_return"             
    ]
    
    df_final = df_fact.select(*final_columns)
    
    return df_final

def main():
    df_sales = spark.table(SILVER_SALES_TABLE)
    
    fact_sales_df = build_fact_sales(df_sales)
    
    fact_sales_df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(GOLD_FACT_SALES)
        
    print(f"Success! Fact table created with {fact_sales_df.count()} transaction rows.")

if __name__ == "__main__":
    main()